# Iron man 2, the BC arm — the incumbent, correctly built

`BC` = v3.1's incumbent **as defined**: klein bald pass on the raw photograph → the **V2 cropper subtracts the head** (cranium path → white) → klein edit with V2's prompt. The prior iron man skipped the subtraction and scored `BCA4` (bald head left on) by mistake; this notebook fixes that.

Everything runs here — the cropper modules ride in the bundle and are executed **verbatim**, then validated against 33 references made locally, before a single edit is generated.

**Session 1 must already be on Drive** (`v3_runs/v34_ironman2_*.zip`) — this notebook reuses its inputs and its 56 bald frames. Runtime → **A100**, then **Run all**. ~35 min, ~CAD 0.07.


In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU
SEEDS = [46, 47, 48]          # the iron-man seeds
MATRIX = "matrix.csv"         # the 200-pair matrix
DRIVE_PROJECT_DIR = "Side projects and shi"


In [ ]:
# 2 · OpenCV FIRST, before anything imports cv2 - the V2 cropper needs ximgproc.guidedFilter
!pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip -q install -U opencv-contrib-python-headless
import cv2
assert hasattr(cv2.ximgproc, 'guidedFilter'), (
    "cv2 lacks ximgproc.guidedFilter - a stale cv2 is already loaded in this kernel. "
    "Runtime -> Restart session, then run this cell FIRST.")
print('opencv', cv2.__version__, 'with ximgproc.guidedFilter  OK')


In [ ]:
# 3 · the rest of the stack; bundle from GitHub
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu
import os, sys, zipfile, torch, onnxruntime as ort
import cv2
assert hasattr(cv2.ximgproc, 'guidedFilter'), 'a dependency reinstalled plain opencv - restart and run cell 2 again'
!cd /content && rm -rf bc && wget -q -O bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v33_ironman_bundle.zip && unzip -qo bundle.zip -d bc
%cd /content/bc
sys.path.insert(0, 'lib')
for f in ('lib/run_ironman.py', 'lib/ironman_bc_crop.py', 'lib/garment_crop.py', 'lib/phase3_variants.py', MATRIX):
    assert os.path.exists(f), f'bundle incomplete: {f}'
print('providers:', ort.get_available_providers(), '| gpu:', torch.cuda.get_device_name(0))
print('validation refs shipped:', len([f for f in os.listdir('validation') if f.endswith('__BC.jpg')]))


In [ ]:
# 4 · Drive: klein cache + session 1's inputs and bald frames
import glob
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
cands = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'), os.path.join(BASE, 'hf_cache')]
found = [c for c in cands if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else cands[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(os.environ['HF_HOME'], exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'], '(klein cached)' if found else '(no cache - downloads ~13 GB once)')

s1 = sorted(glob.glob(os.path.join(BASE, 'v3_runs', 'v34_ironman2_*.zip')))
assert s1, 'no session-1 zip (v34_ironman2_*.zip) in Drive v3_runs/'
with zipfile.ZipFile(s1[-1]) as z:
    take = [n for n in z.namelist() if n.startswith('inputs/') or n.endswith('__bald.jpg')]
    z.extractall('run', members=take)
nb = len(glob.glob('run/refs/*__bald.jpg')); ni = len(glob.glob('run/inputs/*.jpg'))
print(f'{os.path.basename(s1[-1])}: {nb} bald frames, {ni} input files')
assert nb >= 56, f'only {nb} bald frames'


In [ ]:
# 5 · load klein once, timed
import klein_local as K
K.load(); K.info()


In [ ]:
# 6 · the head subtraction, here, then validate against the local refs of record
import numpy as np
for f in glob.glob('run/refs/*__BC.jpg'): os.remove(f)   # never trust a pre-existing BC ref
import ironman_bc_crop as C
C.main('run')            # bald frame -> BiRefNet matte x classes -> head subtracted -> white
bad = []
for vp in sorted(glob.glob('validation/*__BC.jpg')):
    stem = os.path.basename(vp)
    a, b = cv2.imread(vp), cv2.imread('run/refs/' + stem)
    if b is None: bad.append((stem, 'missing')); continue
    if abs(a.shape[0]-b.shape[0]) > 8 or abs(a.shape[1]-b.shape[1]) > 8:
        bad.append((stem, f'shape {a.shape[:2]} vs {b.shape[:2]}')); continue
    bb = cv2.resize(b, (a.shape[1], a.shape[0]))
    mad = float(np.abs(a.astype(np.float32) - bb.astype(np.float32)).mean())
    print(f'  validate {stem}: MAD {mad:.2f}')
    if mad > 4.0: bad.append((stem, f'MAD {mad:.2f}'))
assert not bad, f'cropper disagrees with the local refs of record: {bad}'
n = len(glob.glob('run/refs/*__BC.jpg')); assert n >= 56, f'only {n} BC refs'
print(f'{n} head-subtracted BC refs, validated against {len(glob.glob("validation/*__BC.jpg"))} local refs')


In [ ]:
# 7 · the 600 BC edits (resumable), call 2 on the same canvas as the version
import json
import run_ironman as R
R.main(MATRIX, 'testset', limit=None, seeds=SEEDS, arms=("BC",),
       gpu_usd_per_hour=A100_USD_PER_HOUR, stage="bcedit", bc_canvas="fal")
print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))


In [ ]:
# 8 · zip the BC arm to Drive
import shutil, time
name = f"v34_ironman2_bc_{time.strftime('%Y%m%d_%H%M')}"
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/refs'):
        if '__BC' in f or '__bald' in f: z.write('run/refs/' + f, 'refs/' + f)
    for f in os.listdir('run/gen'):
        if '__BC__' in f: z.write('run/gen/' + f, 'gen/' + f)
    for f in os.listdir('run/meta'): z.write('run/meta/' + f, 'meta/' + f)
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True)
shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', name + '.zip'))
print('->', os.path.join(BASE, 'v3_runs', name + '.zip'))
